# MCPZero Demo Walkthrough

This notebook walks through the MCPZero demo step by step. It runs the same attack scenario twice:

1. Baseline mode: direct tool access, no policy enforcement.
2. Protected mode: tool calls routed through the Safe MCP Proxy executor.

The important point is that the model is not trusted to decide whether an action is safe. The executor makes a deterministic decision from structured facts: tool name, capability, provenance taint, side-effect type, and descriptor integrity.

## 0. Set up imports

The notebook can be opened from the repository root or from the `docs/` directory. This cell finds the repo root and adds it to `sys.path`.

In [ ]:
from pathlib import Path
import json
import sys
import tempfile


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "mcpzero" / "demo.py").exists() and (path / "safe_mcp_proxy").exists():
            return path
    raise RuntimeError("Could not find safe-mcp-proxy repo root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(REPO_ROOT)

## 1. Inspect the world manifest

The MCPZero protected run uses `worlds/mcpzero_demo.yaml`. In this world, `send_email` exists and is allowed as a capability, but external side effects from tainted provenance are denied.

In [ ]:
import yaml

manifest_path = REPO_ROOT / "worlds" / "mcpzero_demo.yaml"
manifest = yaml.safe_load(manifest_path.read_text())

print("world_id:", manifest["world_id"])
print("allowed_tools:", manifest["allowed_tools"])
print("capabilities:")
for name, cfg in manifest["capabilities"].items():
    print(f"  {name:15} allowed={cfg.get('allowed')}")
print("capability_definitions:")
for name, cfg in manifest.get("capability_definitions", {}).items():
    locked = [arg for arg, arg_cfg in cfg.get("args", {}).items() if "literal" in arg_cfg.get("valueFrom", {})]
    actor = [arg for arg, arg_cfg in cfg.get("args", {}).items() if "actor_input" in arg_cfg.get("valueFrom", {})]
    print(f"  {name}: base_tool={cfg.get('base_tool')} locked={locked} actor_input={actor}")
print("taint_rules:", manifest["taint_rules"])
print("side_effects:", manifest["side_effects"])

## 2. Load an attack scenario

`email_injection` models an agent reading a web-sourced document and then trying to send the document contents to an attacker-controlled address.

In [ ]:
from mcpzero.runner.interface import load_scenario

scenario = load_scenario("email_injection")

print("name:", scenario.name)
print("type:", scenario.type)
print("source_channel:", scenario.source_channel)
print("expected baseline:", scenario.expected_baseline)
print("expected protected:", scenario.expected_protected)
print()
print(scenario.description)
print()
for i, step in enumerate(scenario.steps, 1):
    print(f"{i}. {step.tool}: {step.payload}")

## 3. Run the baseline

Baseline mode uses `BaselineAgent`. It calls tool handlers directly and labels successful calls as `ALLOW` with rule `no_policy`. This establishes that the attack would work without the proxy.

In [ ]:
from mcpzero.runner.interface import ScenarioRunner


def print_steps(label, steps):
    print(label)
    print("-" * len(label))
    for step in steps:
        print(f"{step['tool']:12} -> {step['decision']:6}  rule={step.get('rule', '')}")
        if step.get("result") is not None:
            print("  result:", step["result"])


baseline = ScenarioRunner(mode="baseline").run(scenario)
print_steps("Baseline run", baseline.steps)

## 4. Show the protected decision inputs

Before executing the protected run, inspect the exact inputs the policy engine sees for each step. The source channel is `web`, so provenance is tainted. The read is not external; the email is external.

In [ ]:
from safe_mcp_proxy.main import build_executor
from safe_mcp_proxy.provenance import Provenance

executor = build_executor(REPO_ROOT, world_id="mcpzero_demo")
provenance = Provenance.from_source(scenario.source_channel)

print("source_channel:", provenance.source_channel)
print("tainted:", provenance.tainted)
print()

for step in scenario.steps:
    tool, capability, side_effect_type, hash_ok = executor._tool_context(step.tool)
    policy = executor.policy_engine.decide(
        tool_name=step.tool,
        capability=capability,
        taint=provenance.tainted,
        side_effect_type=side_effect_type,
        descriptor_hash_valid=hash_ok,
    )
    print(step.tool)
    print("  capability:", capability)
    print("  side_effect_type:", side_effect_type)
    print("  descriptor_hash_valid:", hash_ok)
    print("  decision:", policy.decision.value)
    print("  rule:", policy.rule_hit)

## 5. Run the protected path

Protected mode uses `SafeMCPProxy`, which wraps the normal `safe_mcp_proxy` executor. Safe reads still work. The dangerous external action is blocked because it combines tainted provenance with an external side effect.

In [ ]:
protected = ScenarioRunner(mode="protected").run(scenario)
print_steps("Protected run", protected.steps)

## 6. Compare the runs

The demo passes when an attack succeeds in baseline and the proxy blocks at least one baseline-allowed step in protected mode. Safe steps do not need to be blocked.

In [ ]:
from mcpzero.verdict.engine import compare

verdict = compare(scenario, baseline, protected)
print(json.dumps(verdict.to_dict(), indent=2))

## 7. Demonstrate a scoped partial capability

The world also exposes `send_email_to_self`, a partial version of `send_email`. The base tool still sends email, but the `to` argument is fixed in the world manifest. The agent-visible schema contains only `subject` and `body`; even if the actor sends a `to` field, the registry strips it and injects the literal receiver from the manifest.

In [ ]:
from safe_mcp_proxy.compiler import compile_world_manifest
from safe_mcp_proxy.executor import Executor
from safe_mcp_proxy.policy_engine import PolicyEngine
from safe_mcp_proxy.registry import ToolRegistry

tables = compile_world_manifest(str(manifest_path))
scoped_registry = ToolRegistry.with_mock_tools(
    allowlist=tables["allowlist"],
    capability_defs=tables["capability_definitions"],
)
scoped_policy = PolicyEngine(
    allowlist=tables["allowlist"],
    capability_map=tables["capability_map"],
    approval_required=tables["approval_required"],
)
# Disable external simulation here so the mock email handler reveals the final receiver.
scoped_executor = Executor(
    scoped_registry,
    scoped_policy,
    str(Path(tempfile.mkdtemp(prefix="mcpzero-scoped-")) / "audit.jsonl"),
    simulate_external=False,
)

tool = scoped_registry.get_tool("send_email_to_self")
print("actor-visible schema:")
print(json.dumps(tool.schema, indent=2))
print()

payload = {
    "to": "attacker@external.example.com",
    "subject": "Status",
    "body": "The actor tried to override the receiver.",
}
result = scoped_executor.execute("send_email_to_self", payload, Provenance.from_source("cli"))
print("actor payload:")
print(json.dumps(payload, indent=2))
print()
print("executor result:")
print(json.dumps(result, indent=2))

## 8. Capture a JSONL trace

The demo observer records the side-by-side run as append-only JSONL. This is useful for UI demos, audit review, and replay.

In [ ]:
from mcpzero.observer.observer import ExecutionObserver

trace_dir = Path(tempfile.mkdtemp(prefix="mcpzero-notebook-"))
observer = ExecutionObserver(trace_dir=trace_dir)

baseline_with_trace = ScenarioRunner(mode="baseline").run(scenario, observer=observer)
protected_with_trace = ScenarioRunner(mode="protected").run(scenario, observer=observer)

print("trace:", observer.log_path)
for entry in observer.entries:
    print(json.dumps(entry, sort_keys=True))

## 9. Run every MCPZero scenario

The CLI does this for the full corpus. This cell keeps the same logic visible inside the notebook and prints summary metrics.

In [ ]:
from attacks.loader import ATTACKS_DIR, load_all
from mcpzero.metrics.reporter import to_dict as metrics_to_dict

verdicts = []
for s in load_all(ATTACKS_DIR):
    b = ScenarioRunner(mode="baseline").run(s)
    p = ScenarioRunner(mode="protected").run(s)
    v = compare(s, b, p)
    verdicts.append(v)
    print(f"{s.name:20} demo_pass={v.demo_pass} proxy_blocked={v.proxy_blocked}")

print()
print(json.dumps(metrics_to_dict(verdicts), indent=2))

## 10. What this demonstrates

The protected path does not rely on an LLM noticing prompt injection. It virtualizes the tool reality available to the agent and enforces deterministic rules before side effects happen.

For `email_injection`, the safe read remains available, but the tainted external email is denied. That distinction is the core demo: useful actions can continue, while risky side effects are blocked at the execution boundary.